### 人工审核中间件 HumanInTheLoopMiddleware

在工具真正执行**之前**暂停，把模型请求的工具调用交给人工审核。审核人可以选择 **批准 / 编辑 / 驳回 / 代为回复**，然后再继续执行。

它基于 LangGraph 的 `interrupt()` 实现，因此必须配合：

- `checkpointer`：保存中断时的状态（例如 `InMemorySaver`）
- 恢复时传入 `Command(resume={"decisions": [...]})`，并提供同一个 `thread_id`

#### 构造参数

```python
HumanInTheLoopMiddleware(interrupt_on, *, description_prefix=..., edit_notice=...)
```

- `interrupt_on`（**必填**）：`dict[str, bool | InterruptOnConfig]`，工具名 → 审核策略
  - `True`：允许全部决策（approve / edit / reject / respond）
  - `False`：自动放行，不中断
  - `InterruptOnConfig`：精细配置
    - `allowed_decisions`：允许的决策列表，取值 `approve` / `edit` / `reject` / `respond`
    - `description`：中断时展示给审核者的描述，`str` 或回调 `(tool_call, state, runtime) -> str`
    - `args_schema`：编辑时参数对应的 JSON Schema（可选）
    - `when`：谓词 `(ToolCallRequest) -> bool`，返回 `False` 则对该次调用自动放行
- `description_prefix`：中断描述的公共前缀，默认 `"Tool execution requires approval"`（若工具配置了 `description` 则被覆盖）
- `edit_notice`：人工选择 `edit` 后，附加到工具结果前的提示文本；传 `None` 则不添加

> 没有出现在 `interrupt_on` 里的工具，默认自动放行。

#### 四种决策（`Command(resume=...)` 时传入）

| 决策 | 含义 | 关键字段 |
| --- | --- | --- |
| `approve` | 批准，按原参数执行 | — |
| `edit` | 编辑后执行（可改工具名与参数） | `edited_action: {"name": ..., "args": {...}}` |
| `reject` | 驳回，不执行工具 | `message`（可选原因） |
| `respond` | 人工代替工具作答，跳过执行 | `message` |

恢复方式：`agent.invoke(Command(resume={"decisions": [ ... ]}), config)`

In [10]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """给指定收件人发送邮件。

    :param to: 收件人邮箱
    :param subject: 主题
    :param body: 正文
    """
    return f"邮件已发送给 {to}，主题：{subject}"


# interrupt_on 是必填参数：这里对 send_email 的每次调用都进行人工审核
# checkpointer 用于保存中断状态，之后才能用 Command(resume=...) 恢复
agent = create_agent(
    model=model,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
            description_prefix="Tool execution requires approval",
        )
    ],
    checkpointer=InMemorySaver(),
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


#### 案例：触发中断

In [11]:
config = {"configurable": {"thread_id": "approve-demo"}}

# 第一次调用：模型请求调用 send_email，中间件在工具执行前中断
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "给 alice@example.com 发一封邮件，主题是会议，正文是明天十点开会"}
        ]
    },
    config,
)

# 中断信息在 __interrupt__ 里：待审核的动作 + 审核配置
interrupt = result["__interrupt__"][0].value
print("待审核的动作：", interrupt["action_requests"])
print("审核配置：", interrupt["review_configs"])


待审核的动作： [{'name': 'send_email', 'args': {'to': 'alice@example.com', 'subject': '会议', 'body': '明天十点开会'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'alice@example.com', 'subject': '会议', 'body': '明天十点开会'}"}]
审核配置： [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]


#### 案例 1：approve（批准）

In [3]:
# 用同一个 thread_id 恢复，批准该工具调用（按原参数执行）
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,
)

for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print("工具结果：", m.content, "| status =", m.status)


工具结果： 邮件已发送给 alice@example.com，主题：会议 | status = success


#### 案例 2：reject（驳回）

In [4]:
config = {"configurable": {"thread_id": "reject-demo"}}
agent.invoke(
    {"messages": [{"role": "user", "content": "给 bob@example.com 发邮件，主题=通知，正文=下午三点开会"}]},
    config,
)

# 驳回：工具不会执行，模型会收到一条 status=error 的说明
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "内容不合适，先别发"}]}),
    config,
)

for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print("工具结果：", m.content, "| status =", m.status)


工具结果： User rejected the tool call for `send_email` with reason: 内容不合适，先别发 | status = error


#### 案例 3：edit（编辑后执行）

In [5]:
config = {"configurable": {"thread_id": "edit-demo"}}
agent.invoke(
    {"messages": [{"role": "user", "content": "给 bob@example.com 发邮件，主题=通知，正文=下午三点开会"}]},
    config,
)

# 编辑：用 edited_action 替换工具名和参数后再执行
# 执行结果前会自动加上 edit_notice，提示模型「人工替换了这次调用」
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "to": "boss@example.com",
                            "subject": "通知",
                            "body": "下午三点开会（已修改收件人）",
                        },
                    },
                }
            ]
        }
    ),
    config,
)

for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print("工具结果：", str(m.content)[:150], "| status =", m.status)


工具结果： Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. Th | status = success


#### 案例 4：respond（人工代为作答）

In [6]:
config = {"configurable": {"thread_id": "respond-demo"}}
agent.invoke(
    {"messages": [{"role": "user", "content": "给 carol@example.com 发邮件，主题=提醒，正文=记得带伞"}]},
    config,
)

# respond：不执行工具，直接用人工提供的内容作为工具结果回给模型
result = agent.invoke(
    Command(resume={"decisions": [{"type": "respond", "message": "（人工回复：已通过其他方式通知，无需发邮件）"}]}),
    config,
)

for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print("工具结果：", m.content, "| status =", m.status)


工具结果： （人工回复：已通过其他方式通知，无需发邮件） | status = success


#### 更多配置：allowed_decisions 与自定义 description

In [7]:
# allowed_decisions 限制可选项；description 覆盖默认的中断描述
# description_prefix 是未单独配置 description 时的公共前缀
agent_limited = create_agent(
    model=model,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "请审核这封邮件的发送",
                }
            },
            description_prefix="需要人工确认",
        )
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "limited-demo"}}
result = agent_limited.invoke(
    {"messages": [{"role": "user", "content": "给 x@example.com 发邮件，主题=hi，正文=hello"}]},
    config,
)
interrupt = result["__interrupt__"][0].value
print("description：", interrupt["action_requests"][0]["description"])
print("allowed_decisions：", interrupt["review_configs"][0]["allowed_decisions"])


description： 请审核这封邮件的发送
allowed_decisions： ['approve', 'reject']


#### 更多配置：when 谓词（按条件才中断）

In [8]:
# when 返回 False 时，该次调用自动放行；只有满足条件才中断
# 例：仅当收件人是 @evil.com 时才需要人工审核
agent_when = create_agent(
    model=model,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "reject"],
                    "when": lambda req: req.tool_call["args"].get("to", "").endswith("@evil.com"),
                }
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

# 普通收件人：不中断，直接执行
config = {"configurable": {"thread_id": "when-good"}}
result = agent_when.invoke(
    {"messages": [{"role": "user", "content": "给 good@example.com 发邮件，主题=hi，正文=hello"}]},
    config,
)
print("good@ 是否中断：", "__interrupt__" in result)
print("直接执行结果：", str(result["messages"][-1].content)[:50])


good@ 是否中断： False
直接执行结果： 邮件已发送 ✅

- 收件人：good@example.com
- 主题：hi
- 正文：hello


#### 更多配置：edit_notice

`edit_notice` 默认为一段提示文本，用于说明「人工替换了这次调用」。如果你不希望注入这段提示，可以在构造中间件时传 `edit_notice=None`：

```python
HumanInTheLoopMiddleware(
    interrupt_on={"send_email": True},
    edit_notice=None,   # 编辑后不加额外提示
)
```

#### 多工具：多个 decision 是什么样子的？

当模型在**同一轮**里调用了多个需要审核的工具时，中间件会把它们合并成一个 `HITLRequest`：

- `action_requests`：按模型调用顺序，逐个列出待审核的动作
- `review_configs`：与 `action_requests` 一一对应，每个工具可以有不同的 `allowed_decisions`
- 恢复时 `decisions` 必须与 `action_requests` **数量相同、顺序对应**，否则会抛出 `ValueError`
- 未配置或 `when` 返回 `False` 的工具不会进入这个中断请求

> 如果模型是分两轮调用的，则会先中断一次；批准并执行后，再在下一轮中断第二次。

In [12]:
# 再加一个工具：删除文件
@tool
def delete_file(path: str) -> str:
    """删除指定路径的文件。

    :param path: 文件路径
    """
    return f"已删除文件 {path}"


# 多个工具的审核策略可以不同：send_email 允许全部决策，delete_file 只允许 approve / reject
agent_multi = create_agent(
    model=model,
    tools=[send_email, delete_file],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_file": {"allowed_decisions": ["approve", "reject"]},
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

config_multi = {"configurable": {"thread_id": "multi-demo"}}
result = agent_multi.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "请一次性完成两件事：1) 给 alice@example.com 发邮件，主题=会议，正文=明天十点开会；2) 删除文件 /tmp/old_report.txt。两个操作都直接调用工具，不要问我。",
            }
        ]
    },
    config_multi,
)

# 一次中断里包含多个待审核动作
interrupt = result["__interrupt__"][0].value
print("待审核动作数量：", len(interrupt["action_requests"]))
for i, (action, cfg) in enumerate(
    zip(interrupt["action_requests"], interrupt["review_configs"]), start=1
):
    print(f"[{i}] {action['name']}  args={action['args']}  允许的决策={cfg['allowed_decisions']}")


待审核动作数量： 2
[1] send_email  args={'to': 'alice@example.com', 'subject': '会议', 'body': '明天十点开会'}  允许的决策=['approve', 'edit', 'reject', 'respond']
[2] delete_file  args={'path': '/tmp/old_report.txt'}  允许的决策=['approve', 'reject']


In [13]:
# 恢复时必须给「每个待审核动作」一个决策，且顺序与 action_requests 一致
# 这里：第 1 个（send_email）批准；第 2 个（delete_file）驳回
result = agent_multi.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"},
                {"type": "reject", "message": "这个文件先别删"},
            ]
        }
    ),
    config_multi,
)

# 工具结果按 action_requests 的顺序返回
for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print(f"工具结果：{m.content} | status = {m.status}")


工具结果：User rejected the tool call for `delete_file` with reason: 这个文件先别删 | status = error
工具结果：邮件已发送给 alice@example.com，主题：会议 | status = success


#### 要点回顾

1. `interrupt_on` 必填，未列出的工具默认自动放行。
2. 必须配 `checkpointer`，并用同一个 `thread_id` + `Command(resume=...)` 恢复。
3. 四种决策：`approve` / `edit` / `reject` / `respond`，由 `allowed_decisions` 约束。
4. `when` 可以做到「只在特定条件下才需要人工审核」，避免所有调用都被打断。
5. 中断详情在返回值的 `__interrupt__` 中（`action_requests` + `review_configs`）。
6. 多个工具同时中断时，`decisions` 要按 `action_requests` 的顺序一一对应给出。